# **Titanic Data Preprocessing using Pipeline and ColumnTransformer**

In [1]:
import numpy as np
import pandas as pd

In [2]:
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder
from sklearn.preprocessing import MinMaxScaler
from sklearn.pipeline import Pipeline,make_pipeline
from sklearn.feature_selection import SelectKBest,chi2
from sklearn.tree import DecisionTreeClassifier

In [4]:
df = pd.read_csv("Titanic-Dataset.csv")

In [5]:
df.head()

,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,1,0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.2500,NaN,S
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,71.2833,C85,C
2,3,1,3,"Heikkinen, Miss. Laina",female,26.0,0,0,STON/O2. 3101282,7.9250,NaN,S
3,4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.0,1,0,113803,53.1000,C123,S
4,5,0,3,"Allen, Mr. William Henry",male,35.0,0,0,373450,8.0500,NaN,S


In [6]:
df.drop(columns=['PassengerId','Name','Ticket','Cabin'],inplace=True)

In [7]:
X_train,X_test,y_train,y_test = train_test_split(df.drop(columns = ['Survived']),df['Survived'],test_size=0.2,random_state = 42)

In [8]:
X_train.head()

,Pclass,Sex,Age,SibSp,Parch,Fare,Embarked
331,1,male,45.5,0,0,28.5000,S
733,2,male,23.0,0,0,13.0000,S
382,3,male,32.0,0,0,7.9250,S
704,3,male,26.0,1,0,7.8542,S
813,3,female,6.0,4,2,31.2750,S


In [9]:
# IMPUTATION USING COLUMN_TRANSFORMER
trf1 = ColumnTransformer([('impute_age',SimpleImputer(),[2]),
                         ('impute_embarked',SimpleImputer(strategy = 'most_frequent'),[6])
                         ],remainder = 'passthrough')

In [10]:
#  ONE HOT ENCODING
trf2 = ColumnTransformer([('ohe_sex_embarked' , OneHotEncoder(sparse_output = False,handle_unknown = 'ignore'),[1,3])
                           ], remainder = 'passthrough')

In [11]:
# SCALING
trf3 = ColumnTransformer([
    ('scale',MinMaxScaler(),slice(0,10))
],remainder='passthrough')

In [12]:
# Feature selection
trf4 = SelectKBest(score_func=chi2,k=8)

In [13]:
# train the model
trf5 = DecisionTreeClassifier()

# **Create Pipeline**

In [14]:
pipe = Pipeline([
    ('trf1',trf1),
    ('trf2',trf2),
    ('trf3',trf3),
    ('trf4',trf4),
    ('trf5',trf5),
])

**Pipeline Vs make_pipeline**

Pipeline requires naming of steps, make_pipeline does not.
(Same applies to ColumnTransformer vs make_column_transformer)

In [15]:
pipe.fit(X_train, y_train)

print("Model trained successfully.")

Model trained successfully.


In [16]:
pipe.named_steps

{'trf1': ColumnTransformer(remainder='passthrough',
                   transformers=[('impute_age', SimpleImputer(), [2]),
                                 ('impute_embarked',
                                  SimpleImputer(strategy='most_frequent'),
                                  [6])]),
 'trf2': ColumnTransformer(remainder='passthrough',
                   transformers=[('ohe_sex_embarked',
                                  OneHotEncoder(handle_unknown='ignore',
                                                sparse_output=False),
                                  [1, 3])]),
 'trf3': ColumnTransformer(remainder='passthrough',
                   transformers=[('scale', MinMaxScaler(), slice(0, 10, None))]),
 'trf4': SelectKBest(k=8, score_func=<function chi2 at 0x00000237FFDD9260>),
 'trf5': DecisionTreeClassifier()}

In [17]:
y_pred = pipe.predict(X_test)

In [18]:
from sklearn.metrics import accuracy_score
accuracy_score(y_test,y_pred)

0.7932960893854749

## Conclusion

Successfully built an end-to-end preprocessing pipeline using:

- SimpleImputer
- OneHotEncoder
- MinMaxScaler
- ColumnTransformer
- Pipeline
- Decision Tree Classifier

The trained pipeline can directly preprocess unseen data and make predictions.